## Parsing methods Experimentation

Quantitative Comparison Metrics:
- Accuracy in table and graphical extraction
- Ability to handle complex formatting (Images, tables, equations)
- Preservation of document structure
- Speed of parsing

Evaluation techniques:
- Manual visual inspection
- End-to-End Testing: Run the entire RAG pipeline with different parsing strategies and evaluating the final output.

---
ref: https://www.reddit.com/r/LangChain/comments/1ef12q6/the_rag_engineers_guide_to_document_parsing/


### Quick Start
1. Run cells 2 to 6 in order.
2. Start MLflow UI in terminal:
   - `mlflow ui --backend-store-uri /Users/carlychinsekyi/Downloads/GitHub/papermind/notebooks/mlruns`
3. Review parser artifacts in MLflow UI:
   - Experiment -> Individual runs -> Artifacts
4. Log manual inspection scores:
   - `log_manual_inspection_by_parser("pymupdf4llm", 4, "Good structure, minor table loss", run_ids)`
   - `log_manual_inspection_by_parser("docling", 5, "Best structure preservation", run_ids)`

In [1]:
import glob
import mlflow
import time
from pathlib import Path

#pdfs = glob.glob("../data/raw/*pdf")
#target_filepath = glob.glob("../data/raw/attention-is-all-you-need.pdf")[0]
#target_filepath

target_filepath = "../data/raw/attention-is-all-you-need-full.pdf"
target_file= Path(target_filepath).name
target_file

/Users/carlychinsekyi/Downloads/GitHub/papermind/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'attention-is-all-you-need-full.pdf'

In [2]:
mlflow.set_tracking_uri("file:///Users/carlychinsekyi/Downloads/GitHub/papermind/notebooks/mlruns")
# TODO: use sqlite instead of filestore -> mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Papermind_Parsing_Audit") 

/Users/carlychinsekyi/Downloads/GitHub/papermind/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///Users/carlychinsekyi/Downloads/GitHub/papermind/notebooks/mlruns/573178881146081418', creation_time=1777274304251, experiment_id='573178881146081418', last_update_time=1777274304251, lifecycle_stage='active', name='Papermind_Parsing_Audit', tags={}, trace_location=None, workspace='default'>

In [3]:
import mlflow
import time
from pathlib import Path

def run_parsing_experiment(parser_name, pdf_path, parse_func):
    """
    Standardized wrapper to run a parser and log results to MLflow.
    Returns the MLflow run_id so manual scoring can be done without copy-paste.
    """
    with mlflow.start_run(run_name=f"{parser_name}_{Path(pdf_path).stem}") as run:
        run_id = run.info.run_id

        # 1. Log Metadata
        mlflow.log_param("parser", parser_name)
        mlflow.log_param("file_name", Path(pdf_path).name)

        # 2. Execute & Time
        start_time = time.time()
        try:
            markdown_text = parse_func(pdf_path)
            duration = time.time() - start_time

            # 3. Log Performance Metrics
            mlflow.log_metric("latency_sec", round(duration, 2))
            mlflow.log_metric("char_count", len(markdown_text))

            # 4. Save Artifact (The actual Markdown)
            output_file = f"experiments/outputs/{parser_name}_result.md"
            Path(output_file).parent.mkdir(parents=True, exist_ok=True)
            with open(output_file, "w") as f:
                f.write(markdown_text)
            mlflow.log_artifact(output_file)

            print(f"{parser_name} complete. Open {output_file} to inspect.")
            print(f"Run ID: {run_id}")

        except Exception as e:
            mlflow.set_tag("status", "failed")
            mlflow.log_text(str(e), "error_log.txt")
            print(f"{parser_name} failed: {e}")

        return run_id


def log_manual_inspection(run_id, score, comments):
    """
    Call this after you've looked at the output file.
    score: 1-5 (1=Trash, 5=Perfect)
    """
    with mlflow.start_run(run_id=run_id):
        mlflow.log_metric("manual_visual_score", score)
        mlflow.set_tag("manual_inspection_notes", comments)
        print(f"Logged score {score} for run {run_id}")


def log_manual_inspection_by_parser(parser_name, score, comments, run_ids_dict):
    """Convenience helper: log manual score using parser name instead of run_id."""
    run_id = run_ids_dict.get(parser_name)
    if not run_id:
        raise ValueError(f"No run_id found for parser '{parser_name}'.")
    log_manual_inspection(run_id, score, comments)

In [4]:
import pymupdf4llm
from docling.document_converter import DocumentConverter

def parse_with_pymupdf4llm(pdf_path: str) -> str:
    return pymupdf4llm.to_markdown(pdf_path)


def parse_with_docling(pdf_path: str) -> str:
    converter = DocumentConverter()
    doc = converter.convert(pdf_path).document
    return doc.export_to_markdown()

run_ids = {}
run_ids["pymupdf4llm"] = run_parsing_experiment("pymupdf4llm", target_filepath, parse_with_pymupdf4llm)
run_ids["docling"] = run_parsing_experiment("docling", target_filepath, parse_with_docling)

print("Run IDs:", run_ids)

pymupdf4llm complete. Open experiments/outputs/pymupdf4llm_result.md to inspect.
Run ID: e2bd7f48ded548568c518eef73231af3


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 2422.84it/s]


docling complete. Open experiments/outputs/docling_result.md to inspect.
Run ID: 5f7b155bbe274de0b08564f891726feb
Run IDs: {'pymupdf4llm': 'e2bd7f48ded548568c518eef73231af3', 'docling': '5f7b155bbe274de0b08564f891726feb'}


In [5]:
# After manual review, log scores without copying run_id:
log_manual_inspection_by_parser("pymupdf4llm", 2, "images omitted. sections and equations are not preserved", run_ids)
log_manual_inspection_by_parser("docling", 5, "Best structure preservation, Tables & Equations are preserved. For images, only text in images are preserved", run_ids)

Logged score 2 for run e2bd7f48ded548568c518eef73231af3
Logged score 5 for run 5f7b155bbe274de0b08564f891726feb


---
### Final - Save parsing output

In [6]:
import json

# Save as JSON
output_dir = f"../data/processed/{Path(target_file).stem}.json"
with open(output_dir, "w") as file:
    file.write(json_output)

# Save as md
output_dir = f"../data/processed/{Path(target_file).stem}.md"
with open(output_dir, "w", encoding="utf-8") as file:
    file.write(md_output)

NameError: name 'json_output' is not defined